In [1]:
import angorapy as ap
import gymnasium as gym
import numpy as np

In [2]:
class MyTask(gym.Env):

    def __init__(self):
        super().__init__()

        self.action_space = gym.spaces.Discrete(4, start=0)  # up, down, left, right
        self.observation_space = gym.spaces.Box(low=0, high=1, shape=(4,), dtype=int)  # xy of target, xy of agent

        self.agent_position = np.array([5, 5])
        self.goal_position = self._sample_goal()

    def _sample_goal(self):
        possible_coords = np.concatenate([np.arange(4), np.arange(7, 10)])

        return np.random.choice(possible_coords, size=2)

    def _get_obs(self):
        return np.concatenate([self.goal_position, self.agent_position])

    def reset(self, **kwargs):
        self.agent_position = np.array([5, 5])
        self.goal_position = self._sample_goal()

        return self._get_obs(), {}

    def step(self, action):
        assert action in range(4)

        if action == 0:
            new_pos = self.agent_position[0]
            new_pos += 1

            self.agent_position[0] = min(new_pos, 10)
        elif action == 1:
            new_pos = self.agent_position[0]
            new_pos -= 1

            self.agent_position[0] = max(new_pos, 0)
        elif action == 2:
            new_pos = self.agent_position[1]
            new_pos += 1

            self.agent_position[1] = min(new_pos, 10)
        elif action == 3:
            new_pos = self.agent_position[1]
            new_pos -= 1

            self.agent_position[1] = max(new_pos, 0)

        reward = -0.5 - np.linalg.norm(self.agent_position - self.goal_position)

        done = False
        if np.all(np.equal(self.agent_position, self.goal_position)):
            reward = 5
            done = True

        return self._get_obs(), reward, done, done, {}


gym.envs.register(
    id=f'MyTask-v0',
    entry_point=MyTask,
    kwargs={},
)

In [3]:
env = ap.make_task("MyTask-v0")

state, info = env.reset()
for episode in range(5):
    for i in range(10):
        obs, reward, done, _, _ = env.step(env.action_space.sample())
        print(obs)       

        if done:
            break   

{'vision': None, 'touch': None, 'proprioception': array([0.00893525, 0.00993823, 0.00969854, 0.0098039 ], dtype=float32), 'goal': None, 'asymmetric': None}
{'vision': None, 'touch': None, 'proprioception': array([ 0.00631191,  0.00702696, -0.99835116,  0.00693108], dtype=float32), 'goal': None, 'asymmetric': None}
{'vision': None, 'touch': None, 'proprioception': array([ 0.00514854,  0.00573714, -0.7062401 ,  1.4114976 ], dtype=float32), 'goal': None, 'asymmetric': None}
{'vision': None, 'touch': None, 'proprioception': array([ 0.00445434,  0.00496821, -0.57672393, -0.5759572 ], dtype=float32), 'goal': None, 'asymmetric': None}
{'vision': None, 'touch': None, 'proprioception': array([ 0.00398014,  0.00444343, -1.5806627 , -0.49886996], dtype=float32), 'goal': None, 'asymmetric': None}
{'vision': None, 'touch': None, 'proprioception': array([ 0.00362976,  0.00405603, -1.7675991 , -0.4462438 ], dtype=float32), 'goal': None, 'asymmetric': None}
{'vision': None, 'touch': None, 'propriocept

/Users/weidler/workspace/angorapy-tutorials/.venv/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:188: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/Users/weidler/workspace/angorapy-tutorials/.venv/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:188: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


In [4]:
model_builder = ap.models.get_model_builder("simple", "ffn")
agent = ap.Agent(model_builder, env)
agent.drill(5, 10, 512)



Drill of agent 1782221707676066 started using 1 processes for 8 workers of which 1 are optimizers. Worker distribution: [8].
IDs over Workers: [[0, 1, 2, 3, 4, 5, 6, 7]]
IDs over Optimizers: [[0, 1, 2, 3, 4, 5, 6, 7]]
Gathering cycle 0...

Before Training; r: -1655.84; len:   241.67; n:  21; loss: [  pi  |  v     |  ent ]; upd:      0; y.exp: 0.000; ; time:  ; time left: unknown time; took s [unknown time left]
Gathering cycle 1...

Cycle⠀     1/5; r:  -909.77; len:   165.64; n:  14; loss: [ -0.02|    0.03|  1.37]; upd:    160; ; time: [11.2|0.0|1.3] [89|0|11]; time left: 0.8mins; took 12.72s [0.8mins left]
Gathering cycle 2...

Cycle⠀     2/5; r:  -895.42; len:   175.88; n:  17; loss: [ -0.02|    0.01|  1.36]; upd:    320; ; time: [11.3|0.0|1.2] [91|0|9]; time left: 0.6mins; took 12.29s [0.6mins left]
Gathering cycle 3...

Cycle⠀     3/5; r:  -406.95; len:    94.67; n:  72; loss: [  0.01|    0.01|  1.35]; upd:    480; ; time: [11.0|0.0|1.1] [91|0|9]; time left: 0.4mins; took 12.67s [0.4mins left]
Gathering cycle 4...

Cycle⠀     4/5; r:  -208.97; len:    56.24; n: 143; loss: [ -0.01|    0.02|  1.34]; upd:    640; ; time: [11.5|0.0|1.3] [90|0|10]; time left: 0.2mins; took 12.6s [0.2mins left]
Finalizing...Drill finished after 62.98serialization.


PPOAgent[at 4][MyTask-v0]